# Pipeline — chạy archive rồi purge

Chạy sau khi `run_setup.ipynb` đã tạo xong resource. Bốn giai đoạn, theo đúng thứ tự:

| Giai đoạn | Hàm | Kết quả |
|---|---|---|
| 1. Archive | `archive.start()`, `archive.status()` | Raw Parquet dưới `S3_PREFIX` |
| 2. Repartition | `partition_initial.run_partition_job()`, `status_partition_job()` | Curated Hive style + Glue Catalog |
| 3. Verify | `purge_source.verify()` | So RDS với archive qua Athena |
| 4. Purge | `purge_source.purge()`, `status()`, `vacuum()` | Xóa dữ liệu cũ khỏi RDS |

Không nhảy bước. Giai đoạn 4 chỉ chạy khi giai đoạn 3 báo `PASSED`, và `purge()` cũng tự chặn nếu verify không PASSED.

In [1]:
%pip install boto3 psycopg2-binary python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if not (notebook_dir / 'archive.py').is_file():
    notebook_dir = Path.cwd() / 'dms'


def load(name):
    """Nạp lại module từ ổ đĩa để không dùng bản cũ còn trong kernel."""
    module_path = (notebook_dir / f'{name}.py').resolve()
    sys.modules.pop(name, None)
    spec = importlib.util.spec_from_file_location(name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Không thể load module từ {module_path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    print(f'Loaded: {module.__file__}')
    return module


archive = load('archive')
partition_initial = load('partition_initial')
purge_source = load('purge_source')

cfg = archive.config()
pcfg = partition_initial.config()
print(f'Raw     : s3://{cfg.s3_bucket}/{cfg.s3_prefix}/{cfg.source_schema}/{cfg.source_table}/')
print(f'Curated : s3://{pcfg.bucket}/{pcfg.curated_prefix}/year=YYYY/month=MM/day=DD/')
print(f'Catalog : {pcfg.glue_database}.{purge_source.config().curated_table}')

Loaded: D:\Project để phỏng vấn\Archive\Archiver-Data\dms\archive.py
Loaded: D:\Project để phỏng vấn\Archive\Archiver-Data\dms\partition_initial.py
Loaded: D:\Project để phỏng vấn\Archive\Archiver-Data\dms\purge_source.py
Raw     : s3://my-data-lake-archival-demo/raw/rds/orders/public/orders/
Curated : s3://my-data-lake-archival-demo/curated/rds/orders/year=YYYY/month=MM/day=DD/
Catalog : archive.orders_rds_orders


## 1. Archive: start full load

`start()` tính lại cutoff ngay trước khi start, nên retention tính theo thời điểm chạy thật. Task đang chạy hoặc đã hoàn tất sẽ không bị start lại — `TargetTablePrepMode=DO_NOTHING` nên replay có thể tạo file trùng trên S3.

In [3]:
archive.start()

START — full load
  Đang chờ task sẵn sàng sau khi cập nhật mapping...
  Đang chờ task sẵn sàng sau khi cập nhật mapping...
  Đang chờ task sẵn sàng sau khi cập nhật mapping...
  Đang chờ task sẵn sàng sau khi cập nhật mapping...
✓ Đã bắt đầu full load: closed_at_utc <= 2026-05-22 17:05:33.000
Chạy status() để theo dõi tiến độ.


'2026-05-22 17:05:33.000'

Chạy lại cell dưới nhiều lần cho tới khi `Task: stopped`, `errors=0` và table statistics có số row hợp lý. Hàm chỉ đọc trạng thái, không restart task.

In [4]:
archive.status()

STATUS — DMS full load
DMS instance : available
RDS endpoint: successful
S3 endpoint : successful
Task         : starting
Tiến độ      : 0% | completed=0 | loading=0 | queued=0 | errors=0
S3           : s3://my-data-lake-archival-demo/raw/rds/orders/
S3 objects   : hiển thị 1 file đầu tiên
  - raw/rds/orders/_pipeline/glue_partition_initial.py (1,967 bytes)


## 2. Repartition sang curated Hive style

DMS full load chỉ là raw landing: nó không tạo folder `year=/month=/day=` theo `DATE_COLUMN`. Glue job đọc raw, partition theo ngày nghiệp vụ rồi crawler cập nhật Data Catalog.

`wait=False` để notebook không bị giữ lâu; cell status bên dưới sẽ tự start và chờ crawler khi Glue job `SUCCEEDED`. Chạy bước này **trước khi** bật `rds_daily_pipeline`.

In [5]:
run_id = partition_initial.run_partition_job(wait=False)

Started Glue run: jr_cfb49872d21d4e9e6fb9ae96be0d9e523b91002a154f02ab6996648c906d890a


In [6]:
partition_initial.status_partition_job(run_id)

Run   : jr_cfb49872d21d4e9e6fb9ae96be0d9e523b91002a154f02ab6996648c906d890a
State : RUNNING


Nếu Glue job đã `SUCCEEDED` nhưng crawler lỗi, cell dưới repair IAM role và retry riêng crawler; không chạy lại repartition.

In [7]:
# partition_initial.retry_crawler()
# partition_initial.crawler_status()

## 3. Verify trước khi xóa

So RDS với curated archive trong đúng phạm vi cutoff của DMS task: số row, số distinct key và tổng `amount`. Chỉ đọc, không xóa. Cần quyền Athena và `psycopg2`; RDS security group phải cho phép IP hiện tại của máy chạy notebook.

In [12]:
import boto3

glue = boto3.client("glue", region_name="ap-southeast-1")

table = glue.get_table(
    DatabaseName="archive",
    Name="orders_rds_orders_22d340054f48ca14bb0c454fd7d79892"
)

print(table["Table"]["StorageDescriptor"]["Location"])

s3://my-data-lake-archival-demo/curated/rds/orders/


In [14]:
report = purge_source.verify()

Cutoff        : 2026-05-22 17:05:33.000 (UTC)
RDS           : rows=629 keys=629 amount=315907.90
Archive       : rows=629 keys=629 amount=315907.90
Verify        : PASSED


## 4. Purge dữ liệu cũ khỏi RDS

Dry run đếm chính xác những gì sẽ bị xóa mà không chạy DELETE. Row nào chưa có trong archive được báo `skipped` và giữ nguyên.

In [15]:
purge_source.purge(dry_run=True)

Cutoff        : 2026-05-22 17:05:33.000 (UTC)
RDS           : rows=629 keys=629 amount=315907.90
Archive       : rows=629 keys=629 amount=315907.90
Verify        : PASSED
Đã tải 629 key từ archive
Child rows sẽ xóa cùng parent: order_items.order_id
DRY RUN, không xóa gì: 629 row orders, 0 child row


{'cutoff': '2026-05-22 17:05:33.000',
 'dry_run': True,
 'deleted_rows': 629,
 'deleted_child_rows': 0,
 'skipped_rows': 0}

Cell dưới xóa thật: theo batch nhỏ, commit từng batch, child rows trước parent để không vi phạm foreign key. **Không hoàn tác được** — chỉ bỏ comment khi dry run đúng như mong đợi và RDS đã có snapshot.

In [19]:
purge_source.purge(dry_run=False)

Cutoff        : 2026-05-22 17:05:33.000 (UTC)
RDS           : rows=629 keys=629 amount=315907.90
Archive       : rows=629 keys=629 amount=315907.90
Verify        : PASSED
Đã tải 629 key từ archive
Child rows sẽ xóa cùng parent: order_items.order_id
batch 1: deleted=500 child=1500
batch 2: deleted=629 child=1887
Đã xóa: 629 row orders, 1887 child row
Lưu ý: allocated storage của RDS không tự thu nhỏ. Chạy vacuum() để giảm bloat và cập nhật statistics.


{'cutoff': '2026-05-22 17:05:33.000',
 'dry_run': False,
 'deleted_rows': 629,
 'deleted_child_rows': 1887,
 'skipped_rows': 0}

In [20]:
purge_source.status()

Cutoff            : 2026-05-22 17:05:33.000 (UTC)
Còn trong cutoff  : 0 row orders
Tổng orders      : 92 row
Tổng order_items : 276 row


{'cutoff': '2026-05-22 17:05:33.000',
 'remaining_in_cutoff': 0,
 'row_counts': {'orders': 92, 'order_items': 276}}

`vacuum()` giảm bloat và refresh statistics sau khi xóa. Allocated storage của RDS instance vẫn không tự thu nhỏ; muốn giảm thật thì phải right-size instance.

In [21]:
purge_source.vacuum()

VACUUM ANALYZE xong: public.orders
VACUUM ANALYZE xong: public.order_items
VACUUM chỉ giải phóng space trong file dữ liệu; allocated storage của RDS instance vẫn không giảm.
